# <b>Coursework 2: Final project report: Experiments & Evaluations</b>

## <b>Research topic: </b> Cross-Lingual Semantic Information Retrieval for Multilingual Humanitarian Platforms using Transformer-based embedding alignments and Low-resource language optimization.

### <b>Submitted by:</b> Himanshu Sharma (SN. 230110847), University of London

*****

## Ingestion & Embedding

In [ ]:
import numpy as np
import pandas as pd

import app.clir_executor as Executer

In [ ]:
# Parameters.
config_path = "config/005-exp-faiss-labse-dense.yml"

In [ ]:
Executer.execute(config_path=config_path, skip_ingest=True, skip_embed=True, skip_index=True)

*****

# Query results retrieval

In [ ]:
# ============================
# 1. Setup
# ============================

import faiss
import numpy as np
from elasticsearch import Elasticsearch
from typing import List, Dict
import matplotlib.pyplot as plt
from collections import defaultdict

In [ ]:
# Create instance of elasticsearch server.
es = Elasticsearch(
    "http://localhost:9200",
    basic_auth=("elastic", "DfZP9TzO")   # 👈 add this
)

In [ ]:
es_index = "clir_docs"

In [ ]:
query_text = "Child protection"

In [ ]:
resp = es.search(
    index=es_index,
    body={"query": {"match": {"text": query_text}}},
    size=20,
)

In [ ]:
hits = [(hit["_id"], hit["_score"]) for hit in resp["hits"]["hits"]]

In [ ]:
hits

In [ ]:
# Load FAISS index
faiss_index = faiss.read_index("artifacts/indexes/xlmr-indexed.faiss")  # update path

In [ ]:








    
hits = [(hit["_id"], hit["_score"]) for hit in resp["hits"]["hits"]]

In [ ]:
# ============================
# 2. Query and Ground Truth
# ============================

# Example multilingual queries
queries = {
    "q1": {"text": "What is the capital of France?", "lang": "en"},
    "q2": {"text": "¿Cuál es la capital de Francia?", "lang": "es"},
    "q3": {"text": "Quelle est la capitale de la France?", "lang": "fr"},
}


In [ ]:
# Example ground truth (query_id -> relevant doc_ids)
ground_truth = {
    "q1": ['vietnam__textcontent_article:701'],  # doc_id 101 is relevant
    "q2": ['vietnam__textcontent_article:701'],
    "q3": ['vietnam__textcontent_article:701'],
}

In [ ]:
# ============================
# 3. Retrieval Functions
# ============================

def faiss_search(query_emb: np.ndarray, top_k=10):
    """Dense search using FAISS."""
    D, I = faiss_index.search(query_emb.astype("float32"), top_k)
    return I[0].tolist(), D[0].tolist()


def es_search(query_text: str, top_k=10):
    """Sparse search using Elasticsearch BM25."""
    resp = es.search(
        index=es_index,
        body={"query": {"match": {"text": query_text}}},
        size=top_k,
    )
    hits = [(hit["_id"], hit["_score"]) for hit in resp["hits"]["hits"]]
    return [str(doc_id) for doc_id, _ in hits], [score for _, score in hits]


def hybrid_fusion(dense_ids, dense_scores, sparse_ids, sparse_scores, method="rrf", alpha=0.5):
    """Fuse dense + sparse results."""
    scores = defaultdict(float)

    if method == "linear":
        for doc, s in zip(dense_ids, dense_scores):
            scores[doc] += alpha * s
        for doc, s in zip(sparse_ids, sparse_scores):
            scores[doc] += (1 - alpha) * s

    elif method == "rrf":
        # Reciprocal Rank Fusion
        for rank, doc in enumerate(dense_ids, 1):
            scores[doc] += 1 / (60 + rank)
        for rank, doc in enumerate(sparse_ids, 1):
            scores[doc] += 1 / (60 + rank)

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    fused_ids = [doc for doc, _ in ranked]
    fused_scores = [score for _, score in ranked]
    return fused_ids, fused_scores


In [ ]:
# ============================
# 4. Evaluation Functions
# ============================

def precision_at_k(results: List[int], relevant: List[int], k=10):
    return len(set(results[:k]) & set(relevant)) / k


def recall_at_k(results: List[int], relevant: List[int], k=10):
    return len(set(results[:k]) & set(relevant)) / len(relevant)


def evaluate(all_results, ground_truth, k=10):
    metrics = {"precision": [], "recall": []}
    for qid, results in all_results.items():
        relevant = ground_truth.get(qid, [])
        metrics["precision"].append(precision_at_k(results, relevant, k))
        metrics["recall"].append(recall_at_k(results, relevant, k))
    return {m: np.mean(vals) for m, vals in metrics.items()}


In [ ]:
# ============================
# 5. Run Experiments
# ============================

dense_results = {}
sparse_results = {}
hybrid_results = {}


for qid, q in queries.items():
    # Dummy embeddings: replace with your encoder
    query_emb = np.random.rand(1, faiss_index.d) 
    print(query_emb) 

    # Dense
    d_ids, d_scores = faiss_search(query_emb)
    dense_results[qid] = d_ids

    # Sparse
    s_ids, s_scores = es_search(q["text"])
    sparse_results[qid] = s_ids

    # Hybrid
    h_ids, h_scores = hybrid_fusion(d_ids, d_scores, s_ids, s_scores, method="rrf")
    hybrid_results[qid] = h_ids

# Evaluate
dense_eval = evaluate(dense_results, ground_truth)
sparse_eval = evaluate(sparse_results, ground_truth)
hybrid_eval = evaluate(hybrid_results, ground_truth)

print("Dense:", dense_eval)
print("Sparse:", sparse_eval)
print("Hybrid:", hybrid_eval)

In [ ]:




# ============================
# 6. Visualization
# ============================

methods = ["Dense", "Sparse", "Hybrid"]
precisions = [dense_eval["precision"], sparse_eval["precision"], hybrid_eval["precision"]]
recalls = [dense_eval["recall"], sparse_eval["recall"], hybrid_eval["recall"]]

x = np.arange(len(methods))
width = 0.35

fig, ax = plt.subplots()
ax.bar(x - width/2, precisions, width, label="Precision@k")
ax.bar(x + width/2, recalls, width, label="Recall@k")

ax.set_ylabel("Score")
ax.set_title("Dense vs Sparse vs Hybrid Retrieval")
ax.set_xticks(x)
ax.set_xticklabels(methods)
ax.legend()
plt.show()


### Elastic BM25 example

In [17]:
from app.clir_config import Config
from app.clir_query import Retriever

from app.indexing.elastic import ElasticClient
import pandas as pd

In [18]:
cfg = Config.load("config/001-exp-es-bm25-sparse.yml")

In [19]:
es = ElasticClient(
    host=cfg.elastic.host,
    username=cfg.elastic.username,
    password=cfg.elastic.password,
    #api_key=cfg.elastic.api_key,
    #verify_certs=cfg.elastic.verify_certs,
)

In [20]:
rt = Retriever(
    meta_path=cfg.paths.metadata_table,
    es=es,
    es_index_name='clir_docs'
)

In [22]:
queries = ["cholera response in Malawi", "cash assistance for refugees in Lebanon"]
results = rt.run(method="elastic_bm25", Q_texts=queries, top_k=20)

In [23]:
results

[[{'doc_id': 'global__textcontent_press_release:3310', 'score': 29.740429},
  {'doc_id': 'global__textcontent_article:1051', 'score': 29.45312},
  {'doc_id': 'global__textcontent_article:1052', 'score': 29.45312},
  {'doc_id': 'global__textcontent_article:1053', 'score': 29.45312},
  {'doc_id': 'global__textcontent_article:1054', 'score': 29.45312},
  {'doc_id': 'global__textcontent_article:1055', 'score': 29.45312},
  {'doc_id': 'global__textcontent_press_release:351', 'score': 24.005577},
  {'doc_id': 'global__textcontent_press_release:3311', 'score': 23.761232},
  {'doc_id': 'global__textcontent_press_release:3659', 'score': 23.638948},
  {'doc_id': 'global__textcontent_press_release:3676', 'score': 23.019794},
  {'doc_id': 'global__textcontent_press_release:3312', 'score': 19.92323},
  {'doc_id': 'global__textcontent_press_release:96', 'score': 19.913002},
  {'doc_id': 'global__textcontent_press_release:3386', 'score': 19.727823},
  {'doc_id': 'global__textcontent_article:1370', 's

In [24]:
meta = pd.read_parquet(cfg.paths.metadata_table)

In [25]:
id2title = dict(zip(meta["id"].astype(str), meta["title"].astype(str)))

In [26]:
for i, q in enumerate(queries):
    print(f"\nQuery: {q}")
    rows = [{"doc_id": r["doc_id"], "score": round(r["score"], 4), "title": id2title.get(r["doc_id"], "")}
            for r in results[i]]
    display(pd.DataFrame(rows))


Query: cholera response in Malawi


,doc_id,score,title
0,global__textcontent_press_release:3310,29.7404,Geneva Palais briefing on the cholera outbreak...
1,global__textcontent_article:1051,29.4531,Response and prevention: battling cholera in H...
2,global__textcontent_article:1052,29.4531,Response and prevention: battling cholera in H...
3,global__textcontent_article:1053,29.4531,Response and prevention: battling cholera in H...
4,global__textcontent_article:1054,29.4531,Response and prevention: battling cholera in H...
5,global__textcontent_article:1055,29.4531,Response and prevention: battling cholera in H...
6,global__textcontent_press_release:351,24.0056,UNICEF statement on Saudi contribution to Yeme...
7,global__textcontent_press_release:3311,23.7612,WHO and UNICEF launch cholera vaccination camp...
8,global__textcontent_press_release:3659,23.6389,As cholera cases continue to rise in parts of ...
9,global__textcontent_press_release:3676,23.0198,UNICEF strengthens response to DRC's worst flo...



Query: cash assistance for refugees in Lebanon


,doc_id,score,title
0,eca__textcontent_article:1153,32.2795,Cash assistance for struggling families in Taj...
1,eca__textcontent_article:1154,32.2795,Cash assistance for struggling families in Taj...
2,eca__textcontent_article:1155,32.2795,Cash assistance for struggling families in Taj...
3,eca__textcontent_article:1156,32.2795,Cash assistance for struggling families in Taj...
4,armenia__textcontent_press_release:238,29.0487,"UNICEF to support families of 3,055 children w..."
5,global__textcontent_article:35,26.3620,Cash assistance helps Syrian children in Iraq ...
6,global__textcontent_article:36,26.3620,Cash assistance helps Syrian children in Iraq ...
7,global__textcontent_article:37,26.3620,Cash assistance helps Syrian children in Iraq ...
8,eca__textcontent_article:58,24.3136,Syrian crisis takes tragic toll on refugee fam...
9,eca__textcontent_press_release:632,24.1488,"UNICEF reaches 350,000 children with cash assi..."


### Dense only.

In [1]:
from app.clir_config import Config
from app.clir_query import QueryEncoder, Retriever
import pandas as pd

In [2]:
cfg = Config.load("config/003-exp-faiss-xlm-r-dense.yml")  # or your config

In [3]:
cfg.model_name()

'sentence-transformers/paraphrase-xlm-r-multilingual-v1'

In [4]:
cfg.embed.normalize

True

In [5]:
# Encoder for query vectors.
qe = QueryEncoder(
    model_name=cfg.model_name(),
    device=cfg.embed.device,          # "cuda" if you have GPU
    batch_size=64,
    normalize=cfg.embed.normalize,
    max_seq_length=256,
)

/Users/himanshusharma/anaconda3/envs/cross-lingual-semantic-search/lib/python3.10/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [6]:
cfg.paths.faiss_index

'artifacts/indexes/xlmr-indexed.faiss'

In [7]:
cfg.paths.embeddings_npy

'artifacts/embeddings/xlmr-embedded.npy'

In [8]:
import faiss, numpy as np, pandas as pd
# Check FAISS index dim
idx = faiss.read_index(cfg.paths.faiss_index)
print("FAISS index.d =", idx.d)

FAISS index.d = 768


In [9]:
# Retriever (needs meta + faiss index)
rt = Retriever(
    meta_path=cfg.paths.metadata_table,
    faiss_index_path=cfg.paths.faiss_index,
)

In [10]:
queries = ["cholera response in Malawi", "cash assistance for refugees in Lebanon"]
Q_vec = qe.encode(queries)  # FAISS needs vectors

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [11]:
Q_vec.shape

(2, 768)

In [12]:
results = rt.run(method="faiss", Q_vecs=Q_vec, top_k=5)  # list[list[{doc_id, score}]]

In [13]:
results

[[{'doc_id': 'global__textcontent_press_release:3180',
   'score': 0.5237458944320679},
  {'doc_id': 'global__textcontent_press_release:794',
   'score': 0.5136306881904602},
  {'doc_id': 'global__textcontent_article:1774', 'score': 0.5107752084732056},
  {'doc_id': 'global__textcontent_press_release:3310',
   'score': 0.5069790482521057},
  {'doc_id': 'china__textcontent_press_release:1099',
   'score': 0.45909735560417175}],
 [{'doc_id': 'armenia__textcontent_press_release:279',
   'score': 0.6143685579299927},
  {'doc_id': 'global__textcontent_press_release:811',
   'score': 0.5859541893005371},
  {'doc_id': 'armenia__textcontent_press_release:239',
   'score': 0.5653582811355591},
  {'doc_id': 'armenia__textcontent_article:1998', 'score': 0.5630447864532471},
  {'doc_id': 'global__textcontent_article:1675', 'score': 0.5613410472869873}]]

In [16]:
id2title = dict(zip(meta["id"].astype(str), meta["title"].astype(str)))
for i, q in enumerate(queries):
    print(f"\nQuery: {q}")
    rows = [{"doc_id": r["doc_id"], "score": round(r["score"], 4), "title": id2title.get(r["doc_id"], "")}
            for r in results[i]]
    display(pd.DataFrame(rows))


Query: cholera response in Malawi


,doc_id,score,title
0,global__textcontent_press_release:3180,0.5237,تفشي الكوليرا يهدد بقاء الأطفال في منطقة الشرق...
1,global__textcontent_press_release:794,0.5136,تنتشر الكوليرا في اليمن بسرعة غير مسبوقة بينما...
2,global__textcontent_article:1774,0.5108,6 وسائل أسهمت بها اليونيسف في تغيير حياة الأفر...
3,global__textcontent_press_release:3310,0.5070,Geneva Palais briefing on the cholera outbreak...
4,china__textcontent_press_release:1099,0.4591,王源探访联合国儿童基金会在马拉维开展的儿童项目



Query: cash assistance for refugees in Lebanon


,doc_id,score,title
0,armenia__textcontent_press_release:279,0.6144,ՄԱԿ-ը 2 միլիոն ԱՄՆ դոլար է տրամադրել Հայաստանի...
1,global__textcontent_press_release:811,0.5860,تعمل مؤسسة كلوني من أجل العدالة بالشراكة مع ال...
2,armenia__textcontent_press_release:239,0.5654,ՅՈՒՆԻՍԵՖ-ը կենսական նշանակության աջակցություն ...
3,armenia__textcontent_article:1998,0.5630,Հագուստի վաուչերներ և դրամական աջակցություն՝ տ...
4,global__textcontent_article:1675,0.5613,في اليمن، 100 دولار يمكنها أن تنقذ حياة إنسان


In [14]:

# Pretty print with titles
meta = pd.read_parquet(cfg.paths.metadata_table)

In [15]:
meta

,id,title,lang
0,armenia__textcontent_article:0,Երբեք չէի պատկերացնի,en
1,armenia__textcontent_article:1,Երբեք չէի պատկերացնի,en
2,armenia__textcontent_article:4,Աղետներին պատրաստ դպրոց` սահմանամերձ գյուղում,hy
3,armenia__textcontent_article:5,Աղետներին պատրաստ դպրոց` սահմանամերձ գյուղում,hy
4,armenia__textcontent_article:6,Աղետներին պատրաստ դպրոց` սահմանամերձ գյուղում,hy
...,...,...,...
21558,vietnam__textcontent_press_release:718,Op-ed: Investing in Futures – Climate Action f...,en
21559,vietnam__textcontent_press_release:719,Op-ed: Investing in Futures – Climate Action f...,en
21560,vietnam__textcontent_press_release:720,Chuyên mục Ý kiến: Đầu tư cho Tương lai – Hành...,vi
21561,vietnam__textcontent_press_release:721,Chuyên mục Ý kiến: Đầu tư cho Tương lai – Hành...,vi


In [ ]:
import app.clir_query as clir_query

In [ ]:
clir_query.run()